In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import skimage.io
import os 
import tqdm
import glob
import tensorflow 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import tensorflow as tf
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.data.experimental import AUTOTUNE
from tensorflow.keras import Sequential, Input, Model
from tensorflow.keras.layers import RandomRotation, RandomZoom
from tensorflow.keras.layers.experimental.preprocessing import Rescaling
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras import applications
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.optimizers import Adam
from keras import layers


from tqdm import tqdm
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from skimage.io import imread, imshow
from skimage.transform import resize

from tensorflow.keras.models import Sequential
from tensorflow.keras.metrics import Precision, AUC,Recall
from tensorflow.keras.layers import InputLayer, BatchNormalization, Dropout, Flatten, Dense, Activation, MaxPool2D, Conv2D
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.applications.densenet import DenseNet169
import copy
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
import cv2
import keras
from keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import load_img, img_to_array
import matplotlib
import matplotlib.pylab as plt
import seaborn as sns
from sklearn.utils import shuffle
from sklearn.metrics import confusion_matrix
from keras.applications.vgg16 import VGG16,preprocess_input
from keras.applications.vgg19 import VGG19,preprocess_input
from tensorflow.keras.utils import image_dataset_from_directory
import tensorflow_addons as tfa
from tensorflow_addons.optimizers import RectifiedAdam
from tensorflow_addons.optimizers import AdamW

labels = pd.read_csv('AlzheimersFLAIR/patient_labels.csv')
labels

In [ ]:
tf.__version__

In [ ]:
def contrast_stretch(image, new_min=0, new_max=1):
    # Compute the current min and max intensity values
    current_min = np.min(image)
    current_max = np.max(image)
    
    # Apply contrast stretching
    stretched_image = (image - current_min) * (new_max - new_min) / (current_max - current_min) + new_min
    
    return stretched_image

In [ ]:
def z_threshold(image):
    mean = np.mean(image)
    std = np.std(image)
    
    image_copy = image.copy()
    z_norm = (image - mean) / std
    
    for i in range(len(z_norm)):
        for j in range(len(z_norm[i])):
            if (z_norm[i][j] >= 4):
                image_copy[i][j] = 0
    return image_copy

In [ ]:
def load_and_preprocess_images(file_path):
    image_data = nib.load(file_path).get_fdata().transpose(2, 0, 1)
    
    preprocessed_data = np.array([np.rot90(contrast_stretch(img), 
                                           k = 1) for img in image_data])[10:-10]
    print(np.shape(preprocessed_data))
    
    
    #print(s[:-7])
    
    return preprocessed_data

In [ ]:
import nibabel as nib

directory = 'Splitted'

x_train = []
y_train = []
x_val = []
y_val = []
x_test = []
y_test = []

used_patients_train = []
used_patients_val = []
used_patients_test = []


for subset in os.listdir(directory):
    if (subset != ".DS_Store"):
        print(subset)
        subset_path = os.path.join(directory, subset)
        for diagnosis in os.listdir(subset_path):
            if (diagnosis != ".DS_Store"):
                diagnosis_path = os.path.join(subset_path, diagnosis)
                for filename in os.listdir(diagnosis_path):
                    print(diagnosis)
                    if (filename != ".DS_Store"):
                        file_path = os.path.join(diagnosis_path, filename)

                        if (subset == 'train'):
                            x_train.append(load_and_preprocess_images(file_path))

                            if (diagnosis == 'CN'):
                                y_train.append(0)
                            elif (diagnosis == 'EMCI'):
                                y_train.append(1)
                            elif(diagnosis == 'LMCI'):
                                y_train.append(2)
                            else:
                                y_train.append(3)
                            
                            used_patients_train.append(filename)

                        elif (subset == 'val'):
                            x_val.append(load_and_preprocess_images(file_path))

                            if (diagnosis == 'CN'):
                                y_val.append(0)
                            elif (diagnosis == 'EMCI'):
                                y_val.append(1)
                            elif(diagnosis == 'LMCI'):
                                y_val.append(2)
                            else:
                                y_val.append(3)
                            
                            used_patients_val.append(filename)
                        else:
                            x_test.append(load_and_preprocess_images(file_path))

                            if (diagnosis == 'CN'):
                                y_test.append(0)
                            elif (diagnosis == 'EMCI'):
                                y_test.append(1)
                            elif(diagnosis == 'LMCI'):
                                y_test.append(2)
                            else:
                                y_test.append(3)
                            
                            used_patients_test.append(filename)

In [ ]:
y_train2 = []
for i in y_train:
    label = [0, 0, 0, 0]
    label[i] += 1
    y_train2.append(label)
y_train2 = np.array(y_train2)

y_val2 = []
for i in y_val:
    label = [0, 0, 0, 0]
    label[i] += 1
    y_val2.append(label)
y_val2 = np.array(y_val2)

y_test2 = []
for i in y_test:
    label = [0, 0, 0, 0]
    label[i] += 1
    y_test2.append(label)
y_test2 = np.array(y_test2)

In [ ]:
x_train2 = np.array(np.expand_dims(x_train, axis = -1))
x_val2 = np.array(np.expand_dims(x_val, axis = -1))
x_test2 = np.array(np.expand_dims(x_test, axis = -1))


# use channels_first instead of channels_last --> that might be the only one compatible with tensorflow GPU


In [ ]:
class CustomDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, volume_list, label_list, batch_size):
        self.volume_tensor = self.preprocess_images(volume_list)
        self.batch_size = batch_size
        self.label_list = label_list

    def __len__(self):
        return int(np.ceil(len(self.volume_tensor) / self.batch_size))

    def __getitem__(self, index):
        batch_indices = self.get_weighted_random_sample()
        batch_x = tf.gather(self.volume_tensor, batch_indices)
        
        batch_labels = [self.label_list[i] for i in batch_indices]
        
        batch_y = []
        for i in batch_labels:
            label = [0, 0, 0, 0]
            label[i] += 1
            batch_y.append(label)
        
        return batch_x, np.array(batch_y)

    def preprocess_images(self, volume_list):
        # Implement your image loading and preprocessing here
        # For example, resizing and normalization
        # Return the batch of preprocessed images as a NumPy array
        batch_data = []
        
        data_augmentor = Sequential([
            layers.RandomRotation(0.1, fill_mode = 'constant'),
            # randomzoom without using 'constant' interpolates with corners of the brain
            # using 'constant' makes randomzoom not actually zoom
        ])
        
        for volume in volume_list:
            padded_volume = []
            for img in volume:
                tensor_img = tf.convert_to_tensor(np.expand_dims(img, axis = -1))
                resized_tensor = tf.image.resize(tensor_img, (128, 128)) # could go lower
                padded_volume.append(data_augmentor(resized_tensor))
            
            padded_volume = tf.convert_to_tensor(padded_volume)
            batch_data.append(padded_volume)
        
        return tf.convert_to_tensor(batch_data).transpose(4, 1, 2, 3, 0)
    
    def get_weighted_random_sample(self):
        class_frequencies = np.bincount(self.label_list)

        # Calculate the inverse of class frequencies as probabilities
        class_probabilities = 1.0 / class_frequencies
        #print(class_probabilities)
        # Normalize the probabilities to sum to 1
        class_probabilities /= np.sum(class_probabilities)
        
        index_probabilities = [class_probabilities[i] for i in self.label_list]
        
        index_probabilities /= np.sum(index_probabilities)
        
        # Randomly choose sample indices with replacement using class probabilities
        random_indices = np.random.choice(np.arange(len(self.label_list)), size=self.batch_size, 
                                          replace=True, p=index_probabilities)
        return random_indices
# Example usage:
# Assuming 'list_of_3d_arrays' is a list of 3D NumPy arrays



In [ ]:
train_generator = CustomDataGenerator(x_train, y_train, batch_size=32)
val_generator = CustomDataGenerator(x_val, y_val, batch_size=32)
test_generator = CustomDataGenerator(x_test, y_test, batch_size=32)

In [ ]:
freq_total = [0, 0, 0, 0]

for x, y in test_generator:
    plt.figure()
    plt.imshow(x[0][10])
    plt.show()
    print(x.shape)
    frequencies = np.bincount(y)
    for i in range(len(frequencies)):
        freq_total[i] += frequencies[i]
    print(y)
print(freq_total)

In [ ]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

In [ ]:
from keras.layers import Conv3D, MaxPooling3D, BatchNormalization, Dropout, Dense, Flatten

conv_layers = Sequential([
    Conv3D(64, (3, 3, 3), activation = 'relu', padding = 'same', name = 'block1_conv1', input_shape = (1, 20, 218, 182),
          data_format = "channels_first"),
    Conv3D(64, (3, 3, 3), activation = 'relu', padding = 'same', name = 'block1_conv2'),
    MaxPooling3D((2, 2, 2), strides = (1, 2, 2), name = 'block1_pool'),
    Conv3D(128, (3, 3, 3), activation = 'relu', padding = 'same', name = 'block2_conv1'),
    Conv3D(128, (3, 3, 3), activation = 'relu', padding = 'same', name = 'block2_conv2'),
    MaxPooling3D((2, 2, 2), strides = (1, 2, 2), name = 'block2_pool'),
    Conv3D(256, (3, 3, 3), activation = 'relu', padding = 'same', name = 'block3_conv1'),
    Conv3D(256, (3, 3, 3), activation = 'relu', padding = 'same', name = 'block3_conv2'),
    Conv3D(256, (3, 3, 3), activation = 'relu', padding = 'same', name = 'block3_conv3'),
    MaxPooling3D((2, 2, 2), strides = (1, 2, 2), name = 'block3_pool'),
    Conv3D(256, (3, 3, 3), activation = 'relu', padding = 'same', name = 'block4_conv1'),
    Conv3D(256, (3, 3, 3), activation = 'relu', padding = 'same', name = 'block4_conv2'),
    Conv3D(256, (3, 3, 3), activation = 'relu', padding = 'same', name = 'block4_conv3'),
    MaxPooling3D((2, 2, 2), strides = (1, 2, 2), name = 'block4_pool')
], 
    name = 'conv_layers')

global_average_layer = tf.keras.layers.GlobalAveragePooling3D(name = 'global_average')

prediction_layer = keras.layers.Dense(4,activation='softmax', name = 'prediction_layer')
#flatten = Flatten()

dense_layers = Sequential([
    BatchNormalization(),
    Dropout(0.3),
    Dense(2048, activation = tf.keras.layers.LeakyReLU(0.1), name = 'dense1'),
    Dropout(0.35),
    Dense(512, activation = tf.keras.layers.LeakyReLU(0.1), name = 'dense2'),
    Dropout(0.3),
    Dense(256, activation = tf.keras.layers.LeakyReLU(0.1), name = 'dense3'),
    Dropout(0.3),
    Dense(64, activation = tf.keras.layers.LeakyReLU(0.1), name = 'dense4')
], 
    name = 'dense_layers')

model = Sequential([
    conv_layers,
    global_average_layer, 
    #flatten,
    dense_layers, 
    prediction_layer
])

model.compile(optimizer=AdamW(learning_rate = 0.001, weight_decay = 0.0001), loss='categorical_crossentropy', metrics=['accuracy',tf.keras.metrics.AUC(),
                        tf.keras.metrics.Precision(),
                        tf.keras.metrics.Recall(),])

In [ ]:
model.summary()

In [ ]:

history=model.fit(train_generator,
                            validation_data=val_generator,
                            steps_per_epoch=len(train_generator),
                            epochs = 15,
                            verbose = 1)
# issue with conv layers (just inputting np arrays instead of data generator doesnt work either)
# using 3d conv is just weird ig
# cpu is too slow to even see if it runs --> just stays at batch 1 epoch 1
# issue with assigning 3d convolutions to gpu